<a href="https://colab.research.google.com/github/Aishwarya23922392/DBATraining/blob/main/Copy_of_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
#This is our Dataset for train our model in the form of array,json,csv,excel,sql.
# ── Dataset ──────────────────────────────────────────────────────────────────
eng_sentences = ["hello", "how are you", "good morning", "thank you", "good night"]

# Add <start> and <end> tokens to Hindi sentences
hin_sentences = [
    "<start> नमस्ते <end>",
    "<start> आप कैसे हैं <end>",
    "<start> सुप्रभात <end>",
    "<start> धन्यवाद <end>",
    "<start> शुभ रात्रि <end>"
]
#Hindi Translation will wrap in between start and and tag.Incoder and Decoder will understand when to start and when to stop.
# ── Tokenizers .It converts given words in integers.
#Example : "Hi"  "How are you"- [2, ,3, 4]
────────────────────────────────────────────────────────────────
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(eng_sentences)
eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)

hin_tokenizer = Tokenizer(filters='')   # keep < > characters
hin_tokenizer.fit_on_texts(hin_sentences)#Fit on text is used to provide a unique new number to every token.
hin_seq = hin_tokenizer.texts_to_sequences(hin_sentences)

max_eng_len = max(len(i) for i in eng_seq)#For equalising the length.
max_hin_len = max(len(i) for i in hin_seq)

#Hii how are you [1,2,3,4]
#hii friends [1,5,0,0]

eng_seq = pad_sequences(eng_seq, maxlen=max_eng_len, padding="post")
hin_seq = pad_sequences(hin_seq, maxlen=max_hin_len, padding="post")

eng_vocab = len(eng_tokenizer.word_index) + 1
hin_vocab = len(hin_tokenizer.word_index) + 1

start_token = hin_tokenizer.word_index["<start>"]
end_token   = hin_tokenizer.word_index["<end>"]

# ── Model ─────────────────────────────────────────────────────────────────────
# Encoder
encoder_inputs    = Input(shape=(None,))
enc_emb           = Embedding(eng_vocab, 64)(encoder_inputs)# Its a dense 64 dimensional vectors.
_, state_h, state_c = LSTM(128, return_state=True)(enc_emb)# emb used for embedding .
encoder_states    = [state_h, state_c]#state_h:It is a hidden state used to remember the sequence of tokens in LSTM. c is a cell state which is used to store the token in long term memory.
#It can combine both states and it will pass to the decoder.
# Decoder
decoder_inputs  = Input(shape=(None,))#used for decoding input layers.
dec_emb_layer   = Embedding(hin_vocab, 64)#embedd the layer for hindi vocabulary.
dec_emb         = dec_emb_layer(decoder_inputs)#it can convert decoder input into hindi word ID.
decoder_lstm    = LSTM(128, return_sequences=True, return_state=True)
dec_out, _, _   = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense   = Dense(hin_vocab, activation="softmax")#DENSE CREATE A WORD MATRIX FOR EVERY WORD.
decoder_outputs = decoder_dense(dec_out)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

# ── Training data ─────────────────────────────────────────────────────────────
decoder_input_data  = hin_seq[:, :-1]          # <start> ... (drop last)
decoder_target_data = hin_seq[:, 1:]           # ... <end>  (drop first)
decoder_target_data = np.expand_dims(decoder_target_data, -1)

model.fit(
    [eng_seq, decoder_input_data],
    decoder_target_data,
    epochs=300,
    batch_size=2,
    verbose=1
)

# ── Inference models ──────────────────────────────────────────────────────────
# Encoder inference
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference
dec_state_input_h = Input(shape=(128,))
dec_state_input_c = Input(shape=(128,))
dec_states_inputs = [dec_state_input_h, dec_state_input_c]

dec_emb2          = dec_emb_layer(decoder_inputs)
dec_out2, h, c    = decoder_lstm(dec_emb2, initial_state=dec_states_inputs)
dec_states2       = [h, c]
dec_dense_out     = decoder_dense(dec_out2)

decoder_model = Model(
    [decoder_inputs] + dec_states_inputs,
    [dec_dense_out]  + dec_states2
)

# ── Translation function ──────────────────────────────────────────────────────
reverse_hin = {v: k for k, v in hin_tokenizer.word_index.items()}

def translate(sentence):
    # Encode input
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding="post")
    states = encoder_model.predict(seq, verbose=0)

    # Start with <start> token
    target_seq = np.array([[start_token]])
    translated = []

    for _ in range(max_hin_len):
        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        word_idx = np.argmax(output[0, -1, :])
        word = reverse_hin.get(word_idx, "")

        if word == "<end>" or word == "":
            break

        translated.append(word)
        target_seq = np.array([[word_idx]])   # next input token
        states = [h, c]                        # update states

    return " ".join(translated)

# ── Test ──────────────────────────────────────────────────────────────────────
for eng in eng_sentences:
    print(f"Input: {eng:15s} → Translated: {translate(eng)}")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │        576 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 64)  │        704 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     98,816 │ embedding[0][0]   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     98,816 │ embedding_1[0][0… │
│                     │ 128), (None,      │            │ lstm[0][1],       │
│                     │ 128), (None,      │            │ lstm[0][2]        │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 11)  │      1,419 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 200,331 (782.54 KB)

 Trainable params: 200,331 (782.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.3000 - loss: 2.3922
Epoch 2/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4000 - loss: 2.3644
Epoch 3/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4000 - loss: 2.3302
Epoch 4/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3500 - loss: 2.2960
Epoch 5/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3500 - loss: 2.2325
Epoch 6/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3500 - loss: 2.1534
Epoch 7/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3500 - loss: 2.0017 
Epoch 8/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3500 - loss: 1.8544 
Epoch 9/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3500 - loss: 1.6240
Epoch 10/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3500 - loss: 1.5877
Epoch 11/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3500 - loss: 1.5406
Epoch 12/300
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3500 - 